In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import types
import pyspark.sql.functions as F

In [ ]:
# Create a spark session
spark = SparkSession.builder.master("local[*]").appName("taxi-rides-app").getOrCreate()

print("[INFO] Starting spark session")

if not spark.version:
    print("[ERROR] Could not start SPARK session. Exiting program.")
    import os

    exit(1)

In [ ]:
from common.config import get_root_path

# Declare dataset paths
DATA_PATH = get_root_path() / "data"
TAXI_PATH = DATA_PATH / "taxi"
DATASET_CLEAN_PATH = TAXI_PATH / "clean" / "yellow" / "2025" / "11"
DATASET_REPORT_PATH = TAXI_PATH / "report"

In [ ]:
print(f"[INFO] Reading clean dataset yellow/2025/11")
dataset_file_path = str(DATASET_CLEAN_PATH)
df = spark.read.parquet(dataset_file_path).repartition(4)

print(f"[INFO] Reading taxi zone lookup dataset")
lookup_file_path = str(DATA_PATH / "taxi_zone_lookup.csv")
lookup_df = spark.read.csv(lookup_file_path, header=True)

In [ ]:
lookup_df.show()

In [ ]:
join_col = df["pickup_location_id"] == lookup_df["LocationID"]
df_joined = df.join(lookup_df, on=join_col)
df_joined.show()

In [ ]:
df2 = df \
    .withColumn("pickup_ts", F.to_timestamp("pickup_datetime")) \
    .withColumn("dropoff_ts", F.to_timestamp("dropoff_datetime")) \
    .withColumn(
        "trip_hours",
        (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 3600.0
    )

# longest trip
df2.orderBy(F.col("trip_hours").desc()).limit(1).select("dropoff_datetime", "pickup_datetime", "trip_hours").show()

In [ ]:
df3 = df_joined.groupBy("Zone").count().orderBy(F.col("count").asc())

df3.show()

In [ ]:
print(f"[INFO] Processing report: homework")

nov_15_rides_count = df.filter(F.extract(F.lit("D"), df.pickup_datetime) == F.lit("15"))

report_df = spark.createDataFrame(
    [
        ("spark-version", spark.version),
        ("parquet-avg-size", 25),
        ("nov_15_rides_count", nov_15_rides_count.count()),
        ("longest_trip_in_hours", 90.6),
        ("spark-ui-port", 4040),
        ("least-frequent-pickup-zone", "Arden Heights")
    ],
    ["name", "value"],
)

In [ ]:
print(f"[INFO] Loading report: homework")
output_path = str(DATASET_REPORT_PATH / "homework")
report_df.repartition(1).write.parquet(
    path=output_path,
    mode="overwrite",
)

In [ ]:
report_df.show()

In [ ]:
# spark.stop()